# 메이플 월드 연구 — 확장 수집 (v5)

두 가지 표본을 추가로 모읍니다.

1. **월드 추가:** 2026-10-28 랭킹 기준, v4에서 모은 5개 월드를 뺀 나머지 9개 월드(제니스·유니온·이노시스·레드·오로라·아케인·노바, 에오스(구 리부트)·핼리오스(구 리부트2)).
2. **이른 랭킹 기준일:** 조회 가능한 가장 이른 날짜 근처(오늘로부터 약 2년 전) 랭킹에서 14개 월드 전체를 다시 표집. 생존 편향 점검용입니다.

**v4 대비 바뀐 점**
- **속도:** 연결 재사용(keep-alive), 동시 요청 96개, 초당 450회(넥슨 한도 500회 미만). 20페이지씩 묶어 랭킹 조회와 캐릭터 조회를 한꺼번에 처리하고, 드라이브 저장도 묶음마다 한 번만 합니다. 예상 소요는 2~3시간입니다.
- **빈 페이지 재시도:** 랭킹이 빈 응답으로 오면 3번까지 다시 묻고, 그래도 비면 완료 처리하지 않습니다.
- **생성 월드 조회:** 조회 날짜를 더 촘촘하게(생성 후 1~7일은 매일) 바꿔 판별 불가를 줄입니다.
- **월드 이름 변경 대응:** 에오스·핼리오스는 2024-12-19 전에는 리부트·리부트2였으므로 이른 기준일에서는 옛 이름으로 조회합니다.

익명 저장 방식(캐릭터명·식별자·순위·페이지 번호 저장 안 함, 레벨은 10단위 구간)은 v4와 같습니다.

**주의:** 이른 기준일 랭킹은 날이 지나면 조회 범위(2년)를 벗어납니다. 계획을 만든 날로부터 **7일 안에** 수집을 끝내세요.

**시작 전:** Colab 보안 비밀 `NEXON_API_KEY`에 서비스 키(`live_`로 시작)를 넣어 주세요.

## 1. 설정

In [ ]:
RECENT_DATE = '2026-10-28'   # v4와 같은 랭킹 기준일
EARLY_MARGIN_DAYS = 7        # 이른 기준일 = 오늘 - 2년 + 이 일수 (수집 기간 동안 조회 가능하도록 여유)

ALL_WORLDS = ['스카니아', '루나', '엘리시움', '크로아', '베라',
              '제니스', '유니온', '이노시스', '레드', '오로라', '아케인', '노바',
              '에오스', '핼리오스']
DONE_IN_V4 = ['스카니아', '루나', '엘리시움', '크로아', '베라']   # 2026-10-28 기준으로 이미 수집함
OLD_NAMES = {'에오스': ['에오스', '리부트'], '핼리오스': ['핼리오스', '리부트2']}   # 2024-12-19 이름 변경

PAGES_PER_WORLD = 400
SEED = 2026
LOOKUP_START, LOOKUP_END = '2023-12-21', '2024-03-31'
LOOKUP_OFFSETS = [1, 2, 3, 4, 5, 6, 7, 10, 14, 21, 30, 45, 60]
LEAF_START = '2024-02-22'

PER_SEC = 450      # 넥슨 서비스 키 한도 500/초보다 낮게
WORKERS = 96       # 동시 요청 수
CHUNK = 20         # 한 번에 처리할 페이지 수
PER_RUN = 15_000_000

## 2. 준비

In [ ]:
import requests, time, random, json, os, csv, uuid, threading
from datetime import date, timedelta
from concurrent.futures import ThreadPoolExecutor
from requests.adapters import HTTPAdapter
import pandas as pd
from google.colab import userdata, drive

drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/maple_main_v5'
os.makedirs(OUT, exist_ok=True)
STATE = f'{OUT}/state.json'

API_KEY = (userdata.get('NEXON_API_KEY') or '').strip()
print('키 앞부분:', API_KEY[:5], '| 길이:', len(API_KEY))
BASE = 'https://open.api.nexon.com/maplestory/v1'

_tls = threading.local()
def session():
    s = getattr(_tls, 's', None)
    if s is None:
        s = requests.Session()
        s.headers['x-nxopen-api-key'] = API_KEY
        s.mount('https://', HTTPAdapter(pool_connections=4, pool_maxsize=4))
        _tls.s = s
    return s

class Budget(Exception): pass
_lock = threading.Lock(); _next = [time.monotonic()]; calls = [0]
def _turn():
    with _lock:
        if calls[0] >= PER_RUN: raise Budget()
        calls[0] += 1
        now = time.monotonic(); slot = max(now, _next[0]); _next[0] = slot + 1 / PER_SEC
    time.sleep(max(0, slot - now))

def get(path, **params):
    last429 = False
    for attempt in range(6):
        _turn()
        try:
            r = session().get(f'{BASE}{path}', params=params, timeout=20)
            d = r.json()
        except Exception:
            time.sleep(min(2 ** attempt, 20)); continue
        if r.status_code == 429 or r.status_code >= 500:
            last429 = r.status_code == 429
            time.sleep(min(2 ** attempt, 20)); continue
        if 'error' in d:
            return {'_error': d['error'].get('name', 'unknown')}
        return d
    if last429: raise Budget()
    return {'_error': 'retry_exhausted'}

def band(level):
    try: l = int(level)
    except (TypeError, ValueError): return ''
    lo = l // 10 * 10
    return f'{lo}-{lo+9}'

def load_state():
    return json.load(open(STATE)) if os.path.exists(STATE) else {}
def save_state(s):
    tmp = STATE + '.tmp'; json.dump(s, open(tmp, 'w'), ensure_ascii=False); os.replace(tmp, STATE)
print('준비 완료')

## 3. 표집 계획 (처음 한 번만 계산)

수집 단위(task) = (랭킹 기준일, 월드). 각 task마다 마지막 페이지를 찾고 무작위 페이지 목록을 정해 `state.json`에 저장합니다. 다시 실행하면 저장된 계획을 그대로 씁니다.

In [ ]:
state = load_state()
if 'tasks' not in state:
    early = (date.today() - timedelta(days=730) + timedelta(days=EARLY_MARGIN_DAYS)).isoformat()
    state = {'early_date': early, 'created': date.today().isoformat(), 'tasks': {}}
    for w in ALL_WORLDS:
        if w not in DONE_IN_V4:
            state['tasks'][f'{RECENT_DATE}|{w}'] = {}
    for w in ALL_WORLDS:
        state['tasks'][f'{early}|{w}'] = {}
    save_state(state)
print('이른 기준일:', state['early_date'], '| 계획 생성일:', state['created'])
print('task 수:', len(state['tasks']))

def rows_of(rank_date, qname, page):
    for k in range(3):   # 빈 응답 재시도
        d = get('/ranking/overall', date=rank_date, world_name=qname, page=page)
        if '_error' in d: return None, d['_error']
        rows = d.get('ranking', [])
        if rows: return rows, None
        time.sleep(1 + k)
    return [], None

def find_last_page(rank_date, qname, cap=2**22):
    rows, err = rows_of(rank_date, qname, 1)
    if err or not rows: return 0
    lo, hi = 1, 2
    while hi <= cap:
        rows, err = rows_of(rank_date, qname, hi)
        if err: raise RuntimeError(f'{qname} {hi}페이지 오류: {err}')
        if not rows: break
        lo, hi = hi, hi * 2
    while hi - lo > 1:
        mid = (lo + hi) // 2
        rows, err = rows_of(rank_date, qname, mid)
        if err: raise RuntimeError(f'{qname} {mid}페이지 오류: {err}')
        if rows: lo = mid
        else: hi = mid
    return lo

for key, t in state['tasks'].items():
    if t.get('pages'): continue
    rank_date, w = key.split('|')
    qname, lp = None, 0
    for cand in OLD_NAMES.get(w, [w]):
        lp = find_last_page(rank_date, cand)
        if lp: qname = cand; break
    rng = random.Random(f'{SEED}-{rank_date}-{w}')
    pages = sorted(rng.sample(range(1, lp + 1), min(PAGES_PER_WORLD, lp))) if lp else []
    rng.shuffle(pages)   # 중간에 멈춰도 표본이 한쪽(상위권)에 치우치지 않도록 순서를 섞음
    state['tasks'][key] = {'world': w, 'rank_date': rank_date, 'query_name': qname, 'last_page': lp, 'pages': pages, 'done': [], 'empty': []}
    save_state(state)
    print(f'{key}: 조회 이름 {qname}, 마지막 페이지 {lp:,} → {len(pages)}페이지 | 누적 호출 {calls[0]:,}')

pd.DataFrame([{'task': k, 'query_name': v['query_name'], 'last_page': v['last_page'], 'pages': len(v['pages']), 'done': len(v['done'])}
              for k, v in state['tasks'].items()])

## 4. 수집 (이어서 실행 가능)

20페이지씩 묶어서 처리합니다. 묶음이 끝날 때마다 `records_<기준일>_<월드>.csv`에 붙이고 진행 상황을 저장합니다. 중간에 끊기면 다시 실행하세요.

In [ ]:
FIELDS = ['rank_date', 'world', 'cluster', 'status', 'same_world', 'date_create',
          'world_first_seen', 'date_first_seen', 'level_band_rank', 'level_band_now']
_YDAY = (date.today() - timedelta(days=1)).isoformat()

def first_seen(ocid, created):
    d0 = date.fromisoformat(created)
    for k in LOOKUP_OFFSETS:
        q = (d0 + timedelta(days=k)).isoformat()
        if q > _YDAY: break
        b = get('/character/basic', ocid=ocid, date=q)
        if '_error' not in b and b.get('world_name'):
            return b['world_name'], q
    return '', ''

def one_char(job):
    rank_date, world, cluster, r = job
    rec = {'rank_date': rank_date, 'world': world, 'cluster': cluster, 'status': '', 'same_world': '', 'date_create': '',
           'world_first_seen': '', 'date_first_seen': '', 'level_band_rank': band(r.get('character_level')), 'level_band_now': ''}
    idr = get('/id', character_name=r.get('character_name'))
    if '_error' in idr:
        rec['status'] = 'id_fail:' + idr['_error']; return rec
    b = get('/character/basic', ocid=idr['ocid'])
    if '_error' in b:
        rec['status'] = 'basic_fail:' + b['_error']; return rec
    rec.update(status='ok', same_world=int(b.get('world_name') in OLD_NAMES.get(world, [world])),
               date_create=(b.get('character_date_create') or '')[:10], level_band_now=band(b.get('character_level')))
    if LOOKUP_START <= rec['date_create'] <= LOOKUP_END:
        rec['world_first_seen'], rec['date_first_seen'] = first_seen(idr['ocid'], rec['date_create'])
    return rec

def fetch_page(job):
    key, t, p = job
    rows, err = rows_of(t['rank_date'], t['query_name'], p)
    return key, p, rows, err

stopped = False; t0 = time.time(); start_calls = calls[0]
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    for key, t in state['tasks'].items():
        if not t.get('query_name'): continue
        todo = [p for p in t['pages'] if p not in set(t['done'])]
        path = f"{OUT}/records_{t['rank_date']}_{t['world']}.csv"
        for i in range(0, len(todo), CHUNK):
            chunk = todo[i:i + CHUNK]
            try:
                pages = list(pool.map(fetch_page, [(key, t, p) for p in chunk]))
                for _, p, rows, err in pages:
                    if err: raise RuntimeError(f'{key} {p}페이지 랭킹 오류: {err}')
                jobs, done_now, empty_now = [], [], []
                for _, p, rows, err in pages:
                    if not rows: empty_now.append(p); continue
                    cl = uuid.uuid4().hex[:10]
                    jobs += [(t['rank_date'], t['world'], cl, r) for r in rows]
                    done_now.append(p)
                recs = list(pool.map(one_char, jobs))
            except Budget:
                stopped = True; break
            random.shuffle(recs)
            new = not os.path.exists(path)
            with open(path, 'a', newline='', encoding='utf-8-sig' if new else 'utf-8') as f:
                wr = csv.DictWriter(f, fieldnames=FIELDS)
                if new: wr.writeheader()
                wr.writerows(recs)
            t['done'] += done_now; t['empty'] = sorted(set(t['empty']) | set(empty_now))
            save_state(state)
            el = time.time() - t0; rate = (calls[0] - start_calls) / max(el, 1)
            print(f"{key}: {len(t['done'])}/{len(t['pages'])}페이지 | 초당 {rate:,.0f}회 | 누적 {calls[0]:,}회 | {el/60:,.1f}분")
        if stopped: break

print('한도 초과로 멈췄습니다. 다시 실행하면 이어서 수집합니다.' if stopped else '모든 task 수집 완료.', '| 이번 실행 호출', f'{calls[0]:,}')

## 5. 진행 상황

In [ ]:
st = load_state()
prog = pd.DataFrame([{'task': k, 'pages': len(v['pages']), 'done': len(v['done']), 'empty': len(v.get('empty', []))} for k, v in st['tasks'].items()])
print(prog.to_string(index=False))
print('\n전체 진행률:', f"{prog.done.sum() / max(prog.pages.sum(), 1):.1%}")